# Set up

In [14]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [5]:
len(documents)

72

In [8]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

# Generating ground truth

In [4]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

# Q1. Generating questions

In [15]:
paginas = {
    '01-agentic-rag/lessons/01-intro.md',
    '01-agentic-rag/lessons/02-environment.md',
    '01-agentic-rag/lessons/03-rag.md',
}

docs = [d for d in documents if d['filename'] in paginas]

In [16]:
import json

user_prompt = json.dumps(docs)

In [17]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [18]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt},
]

In [20]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [26]:
response.usage.input_tokens/3

1243.6666666666667

In [36]:
from evaluation_utils import llm_structured
uso = []
for doc in docs:
    user_prompt = json.dumps(doc)
        
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    uso.append(usage)

print(usage)

ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1867)


In [37]:
uso

[ResponseUsage(input_tokens=1020, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=113, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1133),
 ResponseUsage(input_tokens=1286, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=119, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1405),
 ResponseUsage(input_tokens=1753, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=114, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1867)]

In [39]:
(1020 + 1286 + 1753) / 3

1353.0

# The full ground truth

In [40]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [41]:
len(chunks)

295

In [57]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [60]:
from minsearch import Index
def build_index(doc):
    index = Index(
        text_fields=['content'],
        keyword_fields=["filename"]
    )
    index.fit(doc)
    return index

In [61]:
index = build_index(chunks)

In [69]:
def text_search(query, num_results=5):

    return index.search(
        query,
        num_results=num_results,
    )

In [73]:
#Ahora vectorizamos toda la base de datos

from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# --- 1. Vectorizar TODOS los chunks de una sola vez (batch) ---
print("Encoding chunks...")
texts = [chunk["content"] for chunk in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding chunks...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

In [75]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(embeddings, chunks)

In [76]:
def vector_search(query, num_results=5):
    query_vector = model.encode(query)  # solo la query se vectoriza acá
    return vindex.search(query_vector, num_results=num_results)

In [72]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [43]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

# Q2. First result with text search
# Q3. First result with vector search

In [46]:
import pandas as pd

In [51]:
ground_truth = pd.read_csv("ground-truth.csv")
ground_truth = ground_truth.to_dict(orient="records")

In [53]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [68]:
r_text_search = text_search(q, num_results=5)

In [84]:
r_text_search

[{'start': 3000,
  'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retriev

In [77]:
r_vector_search = vector_search(q, num_results=5)

In [78]:
print("Text search results:")
for r in r_text_search:
    print(f"- {r['filename']} (start: {r['start']})")
print("\nVector search results:")
for r in r_vector_search:
    print(f"- {r['filename']} (start: {r['start']})")

Text search results:
- 01-agentic-rag/lessons/03-rag.md (start: 3000)
- 01-agentic-rag/lessons/13-function-calling.md (start: 1000)
- 01-agentic-rag/lessons/03-rag.md (start: 2000)
- 01-agentic-rag/lessons/13-function-calling.md (start: 2000)
- 01-agentic-rag/lessons/01-intro.md (start: 0)

Vector search results:
- 01-agentic-rag/lessons/01-intro.md (start: 0)
- 04-evaluation/lessons/02-ground-truth.md (start: 0)
- 04-evaluation/lessons/01-intro.md (start: 1000)
- 04-evaluation/lessons/12-rag-answers.md (start: 2000)
- 04-evaluation/lessons/11-evaluation-intro.md (start: 0)


# Evaluation metrics

In [85]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [86]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?


[0, 0, 0, 0, 1]

In [87]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [90]:
relevance_total_text = compute_relevance_total_text(ground_truth)

  0%|          | 0/360 [00:00<?, ?it/s]

Ahora lo generalizamos para cualquier funcion de busqueda

In [94]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [95]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [96]:
relevance_total_vector = compute_relevance_total_text(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

Ahora los evaluamos

# Q4. Evaluating text search

In [97]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [98]:
score = hit_rate(relevance_total_text)

In [99]:
score

0.7583333333333333

# Q5. Evaluating vector search

In [100]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [101]:
mmr = mrr(relevance_total_vector)
mmr

0.6356944444444446

# Q6. Tuning hybrid search

In [103]:
q = ground_truth[0]
r = hybrid_search(q["question"], k=60)
r

[{'start': 0,
  'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour ph

In [106]:
def compute_relevance_hybrid(q, k):
    doc_id = q["filename"]
    results = hybrid_search(query=q["question"], k=k)

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [109]:
from tqdm.auto import tqdm

def compute_relevance_total_text_hybrid(ground_truth,k):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_hybrid(q, k)
        relevance_total.append(relevance)

    return relevance_total

In [111]:
k_values = [1,50,100,200]
mrr_values = []
for k in k_values:
    print(f"Computing MRR for k={k}...")
    relevance_total_text = compute_relevance_total_text_hybrid(ground_truth, k)
    mrr_values.append(mrr(relevance_total_text))
    print(f"MRR for k={k}: {mrr_values[-1]}")
print(mrr_values)

Computing MRR for k=1...


  0%|          | 0/360 [00:00<?, ?it/s]

MRR for k=1: 0.6722685185185188
Computing MRR for k=50...


  0%|          | 0/360 [00:00<?, ?it/s]

MRR for k=50: 0.6721296296296295
Computing MRR for k=100...


  0%|          | 0/360 [00:00<?, ?it/s]

MRR for k=100: 0.6721296296296295
Computing MRR for k=200...


  0%|          | 0/360 [00:00<?, ?it/s]

MRR for k=200: 0.6721296296296295
[0.6722685185185188, 0.6721296296296295, 0.6721296296296295, 0.6721296296296295]
